# Interactive model

In [ ]:
import json
import easyocr
import gradio as gr
import numpy as np
import ollama
import io
import base64
from PIL import Image

reader = easyocr.Reader(['en'], gpu=True)

def pil_to_base64(pil_image):
    """Converts PIL Image to Base64 for Ollama Vision models."""
    buffered = io.BytesIO()
    pil_image = pil_image.convert("RGB")
    pil_image.thumbnail((1024, 1024)) 
    pil_image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode('utf-8')

def get_raw_ocr(pil_image):
    """Standpoint 1: Raw OCR"""
    img_array = np.array(pil_image.convert('RGB'))
    result = reader.readtext(img_array)
    return " \n ".join([res[1] for res in result])

def multimodal_llm_process(pil_image):
    """Standpoint 2: Improved OCR using Llama"""
    img_b64 = pil_to_base64(pil_image)
    
    # Precise prompt to avoid the model complaining about "rotation". Remember that i had an error regarding rotation
    prompt = """Extract all text from this receipt. Focus on the Merchant Name, Date, Address, and individual items. 
    Output the text in a clean, structured list. Do not mention image quality or orientation."""
    
    try:
        response = ollama.chat(model='llama3.2-vision',
            messages=[{'role': 'user', 'content': prompt,'images': [img_b64]}])
        return response['message']['content']
    except Exception as e:
        return f"Multimodal Error: {str(e)}"

def entity_analysis_llm(text_input):
    """Standpoint 3: Entity Analysis"""
    # Safety check: if input is an error message, return a placeholder
    if "Error" in text_input or len(text_input) < 20:
        return {"error": "Insufficient text for analysis"}

    prompt = f"""
    Analyze the receipt text below. 
    1. Fix typos (e.g., 'sollor' -> 'seller', 'T0tal' -> 'Total').
    2. Format into this EXACT JSON: {{"company": "", "date": "", "address": "", "total": ""}}
    3. If a value is missing, use "N/A". Return ONLY the JSON.

    Text:
    {text_input}
    """
    
    try:
        response = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])
        content = response['message']['content']
        # Clean the string to ensure only JSON remains
        start = content.find('{')
        end = content.rfind('}') + 1
        return json.loads(content[start:end])
    except Exception:
        return {"error": "JSON Parse Failed", "raw_content": content[:100]}

def main_pipeline(image, question):
    if image is None: return "Upload an image.", "N/A", {}, {}, "No image provided."

    # ====> 1. Raw OCR
    raw_ocr = get_raw_ocr(image)
    
    # ====>  2. Improved OCR (Multimodal)
    improved_ocr = multimodal_llm_process(image)
    
    # ====>  3. Raw OCR + Entity LLM
    # Rectification of the messy OCR text
    out3 = entity_analysis_llm(raw_ocr)
    
    # ====> 4. Improved OCR + Entity LLM
    # prioritize improved_ocr, but fallback to raw_ocr if the vision model hallucinated/failed
    is_vision_bad = "difficult to read" in improved_ocr.lower() or "rotation" in improved_ocr.lower()
    final_input = raw_ocr if is_vision_bad else improved_ocr
    out4 = entity_analysis_llm(final_input)
    
    # --- Q&A ---
    qa_prompt = f"Using this data: {json.dumps(out4)}, answer this question: {question}"
    qa_res = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': qa_prompt}])
    answer = qa_res['message']['content']
    
    return raw_ocr, improved_ocr, out3, out4, answer

with gr.Blocks(title="SROIE Offline Agent") as demo:
    gr.Markdown("Reading Receipt Informations, let's go!")
    
    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type="pil", label="Receipt Image")
            q_input = gr.Textbox(label="Ask a question", value="What is the total?")
            btn = gr.Button("Process All Standpoints", variant="primary")
            
        with gr.Column():
            with gr.Tabs():
                with gr.TabItem("Raw OCR"):
                    res1 = gr.Textbox(label="EasyOCR Output")
                with gr.TabItem("Multimodal"):
                    res2 = gr.Textbox(label="Llama 3.2 Vision Output")
                with gr.TabItem("Raw OCR + Entity Analysis"):
                    res3 = gr.JSON(label="Rectified Raw OCR")
                with gr.TabItem("Multimodal + Entity Analysis"):
                    res4 = gr.JSON(label="Final Combined Result")
                    res5 = gr.Textbox(label="Q&A Response")

    btn.click(main_pipeline, inputs=[img_input, q_input], outputs=[res1, res2, res3, res4, res5])

demo.launch()

# Model evaluation - raw test

In [ ]:
import os
import pandas as pd
import easyocr
from rapidfuzz import fuzz
from jiwer import wer, cer
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

IMG_DIR = "SROIE2019/test/img"
BOX_DIR = "SROIE2019/test/box" 
LIMIT = 100

reader = easyocr.Reader(['en'], gpu=True)

def evaluate_standpoint_1():
    files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg'))])[:LIMIT]
    scores = {"fuzzy": [], "wer": [], "cer": []}

    print(f"📉 Evaluating Standpoint 1 (Raw EasyOCR) on {len(files)} images...")

    for f_name in tqdm(files):
        try:
            # 1. Get OCR Text
            res = reader.readtext(os.path.join(IMG_DIR, f_name), detail=0, paragraph=True)
            pred_text = " ".join(res).strip().upper()

            # 2. Get Ground Truth Text
            base = os.path.splitext(f_name)[0]
            with open(os.path.join(BOX_DIR, base + ".txt"), 'r') as f:
                # SROIE box files often have coordinates; we just want the text
                gt_text = " ".join([line.split(",")[-1].strip() for line in f.readlines()]).upper()

            if not gt_text: continue

            # 3. Calculate Metrics
            scores["fuzzy"].append(fuzz.ratio(pred_text, gt_text) / 100.0)
            scores["wer"].append(wer(gt_text, pred_text))
            scores["cer"].append(cer(gt_text, pred_text))
        except:
            continue

    # --- Report ---
    final = {
        "Metric": ["Fuzzy Accuracy", "Word Error Rate (WER)", "Character Error Rate (CER)"],
        "Value": [
            f"{sum(scores['fuzzy'])/len(scores['fuzzy']):.2%}",
            f"{sum(scores['wer'])/len(scores['wer']):.2%}",
            f"{sum(scores['cer'])/len(scores['cer']):.2%}"
        ]
    }
    print("\n" + "="*40)
    print(pd.DataFrame(final).to_string(index=False))
    print("="*40)

if __name__ == "__main__":
    evaluate_standpoint_1()

In [3]:
import os, json, base64, io, gc
import ollama
import pandas as pd
from PIL import Image
from rapidfuzz import fuzz
from jiwer import wer, cer
from tqdm import tqdm

# --- CONFIG ---
IMG_DIR = "SROIE2019/test/img"
BOX_DIR = "SROIE2019/test/box" # Using BOX for full text ground truth
LIMIT = 100 

def get_b64(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        img.thumbnail((512, 512)) # Consistent with your working script
        buf = io.BytesIO()
        img.save(buf, format="JPEG")
        return base64.b64encode(buf.getvalue()).decode('utf-8')

def run_evaluation_standpoint_2():
    files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg'))])[:LIMIT]
    results_cache = []
    
    print(f"🚀 Evaluating Standpoint 2: Vision OCR (Resilient Mode)")

    for i, f_name in enumerate(files):
        # Using the exact print style you had
        print(f"[{i+1}/{LIMIT}] Processing: {f_name}...", end="\r")
        try:
            b64 = get_b64(os.path.join(IMG_DIR, f_name))
            
            # Use the same options that worked for you
            res = ollama.generate(
                model='llama3.2-vision', 
                prompt="OCR this receipt. Text only.", 
                images=[b64],
                options={"num_predict": 400} 
            )
            
            # Store the raw text output
            results_cache.append({
                "file": f_name, 
                "pred_text": res['response'].strip().upper()
            })
            
            # Same memory clearing logic
            if i % 5 == 0: gc.collect()

        except Exception as e:
            print(f"\n⚠️ Error on {f_name}: {e}")
            continue

    # --- Phase 2: Calculating OCR Metrics (Fuzzy, WER, CER) ---
    print("\n\n📊 Calculating OCR Accuracy Metrics...")
    scores = {"fuzzy": [], "wer": [], "cer": []}

    for item in tqdm(results_cache):
        try:
            base = os.path.splitext(item['file'])[0]
            gt_path = os.path.join(BOX_DIR, base + ".txt")
            
            if not os.path.exists(gt_path): continue

            # Parse SROIE box files to get ground truth text
            with open(gt_path, 'r') as f:
                gt_text = " ".join([line.split(",")[-1].strip() for line in f.readlines()]).upper()

            pred_text = item['pred_text']
            
            if not gt_text: continue

            # Standard OCR Metrics
            scores["fuzzy"].append(fuzz.ratio(pred_text, gt_text) / 100.0)
            scores["wer"].append(wer(gt_text, pred_text))
            scores["cer"].append(cer(gt_text, pred_text))
        except Exception:
            continue

    # --- Final Report Formatting ---
    final_report = [
        {"Metric": "Fuzzy Accuracy", "Value": sum(scores['fuzzy'])/len(scores['fuzzy'])},
        {"Metric": "Word Error Rate (WER)", "Value": sum(scores['wer'])/len(scores['wer'])},
        {"Metric": "Character Error Rate (CER)", "Value": sum(scores['cer'])/len(scores['cer'])}
    ]
    
    print("\n" + "="*50)
    print("      STANDPOINT 2: VISION OCR RESULTS")
    print("="*50)
    print(pd.DataFrame(final_report).to_string(index=False, formatters={'Value': '{:,.2%}'.format}))
    print("="*50)

if __name__ == "__main__":
    # Ensure Ollama is restarted before running
    run_evaluation_standpoint_2()

🚀 Evaluating Standpoint 2: Vision OCR (Resilient Mode)
[100/100] Processing: X51005724611.jpg...

📊 Calculating OCR Accuracy Metrics...


100%|███████████████████████████████████████| 100/100 [00:00<00:00, 1418.12it/s]


      STANDPOINT 2: VISION OCR RESULTS
                    Metric   Value
            Fuzzy Accuracy  39.18%
     Word Error Rate (WER) 104.81%
Character Error Rate (CER)  89.43%


# Model evaluation - entity

In [ ]:
# import os
import json
import pandas as pd
import ollama
import easyocr
import warnings
from rapidfuzz import fuzz
from tqdm import tqdm

# Suppress hardware/MPS warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torch")

# --- Configuration ---
DATASET_PATH = "SROIE2019"
IMG_DIR = os.path.join(DATASET_PATH, "test", "img")
ENT_DIR = os.path.join(DATASET_PATH, "test", "entities")

# Initialize OCR
reader = easyocr.Reader(['en'], gpu=True)

def evaluate_standpoint_3_combined(limit=100):
    all_images = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg'))])
    test_files = all_images[:limit]
    
    # Storage for both metrics per field
    # fuzzy_scores: floats 0.0-1.0
    # exact_matches: ints 0 or 1
    stats = {
        "company": {"fuzzy": [], "em": []},
        "date": {"fuzzy": [], "em": []},
        "address": {"fuzzy": [], "em": []},
        "total": {"fuzzy": [], "em": []}}
    
    print(f"📊 Evaluating Standpoint 3 on {len(test_files)} images...")
    print(f"Metrics: Fuzzy Similarity & Exact Match (Pass/Fail)\n")

    for filename in tqdm(test_files):
        base_name = os.path.splitext(filename)[0]
        img_path = os.path.join(IMG_DIR, filename)
        gt_path = os.path.join(ENT_DIR, base_name + ".txt")
        
        if not os.path.exists(gt_path):
            continue

        try:
            # 1. OCR Step
            ocr_res = reader.readtext(img_path, detail=0, paragraph=True)
            raw_text = " ".join(ocr_res)

            # 2. LLM Extraction Step
            prompt = f"""Extract to JSON: {{"company": "", "date": "", "address": "", "total": ""}}
            Text: {raw_text}
            Return ONLY JSON."""
            
            response = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])
            content = response['message']['content']
            
            # Parse JSON safely
            json_str = content[content.find('{'):content.rfind('}')+1]
            pred_json = json.loads(json_str)

            # 3. Ground Truth Comparison
            with open(gt_path, 'r') as f:
                gt_json = json.load(f)

            # Calculate metrics for each variable
            for field in stats.keys():
                # Normalization for fair comparison
                p_val = str(pred_json.get(field, "")).strip().upper()
                g_val = str(gt_json.get(field, "")).strip().upper()
                
                # A. Fuzzy Accuracy (Similarity %)
                fuzz_score = fuzz.token_sort_ratio(p_val, g_val) / 100.0
                stats[field]["fuzzy"].append(fuzz_score)
                
                # B. Exact Match (Pass/Fail)
                em_score = 1 if p_val == g_val else 0
                stats[field]["em"].append(em_score)

        except Exception:
            continue

    # --- Generate Combined Report ---
    report_data = []
    for field, data in stats.items():
        avg_fuzzy = sum(data["fuzzy"]) / len(data["fuzzy"]) if data["fuzzy"] else 0
        avg_em = sum(data["em"]) / len(data["em"]) if data["em"] else 0
        
        report_data.append({
            "Variable": field.capitalize(),
            "Fuzzy Accuracy": avg_fuzzy,
            "Exact Match (EM)": avg_em
        })

    report_df = pd.DataFrame(report_data)
    
    print("\n" + "="*60)
    print("      STANDPOINT 3: DUAL-METRIC PERFORMANCE REPORT")
    print("="*60)
    print(report_df.to_string(index=False, formatters={
        'Fuzzy Accuracy': '{:,.2%}'.format,
        'Exact Match (EM)': '{:,.2%}'.format1
    }))
    print("="*60)
    print(f"Processed {len(stats['company']['fuzzy'])} images successfully.")

if __name__ == "__main__":
    evaluate_standpoint_3_combined(limit=100)

In [2]:
import os, json, base64, io, gc
import ollama
import pandas as pd
from PIL import Image
from rapidfuzz import fuzz
from tqdm import tqdm

# --- CONFIG ---
IMG_DIR = "SROIE2019/test/img"
ENT_DIR = "SROIE2019/test/entities"
LIMIT = 100 

def get_b64(path):
    with Image.open(path) as img:
        img = img.convert("RGB")
        # Reducing size further to 512px to ensure it NEVER hangs on memory
        img.thumbnail((512, 512)) 
        buf = io.BytesIO()
        img.save(buf, format="JPEG")
        return base64.b64encode(buf.getvalue()).decode('utf-8')

def run_evaluation():
    files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.jpg', '.jpeg'))])[:LIMIT]
    results_cache = []
    
    print(f"🚀 Starting Resilient Phase 1 (512px Optimized)")

    for i, f_name in enumerate(files):
        print(f"[{i+1}/{LIMIT}] Processing: {f_name}...", end="\r")
        try:
            b64 = get_b64(os.path.join(IMG_DIR, f_name))
            
            # Using generate with a 30-second internal limit to prevent hangs
            res = ollama.generate(
                model='llama3.2-vision', 
                prompt="OCR this receipt. Text only.", 
                images=[b64],
                options={"num_predict": 300} # Limits long-winded hallucinations
            )
            
            # Pass 2: Immediate extraction while model is warm (Hybrid check)
            text = res['response']
            prompt = f"JSON: {{\"company\":\"\",\"date\":\"\",\"address\":\"\",\"total\":\"\"}}. Text: {text}"
            extraction = ollama.chat(model='llama3', messages=[{'role': 'user', 'content': prompt}])
            
            # Save progress immediately
            results_cache.append({"file": f_name, "pred": extraction['message']['content']})
            
            # Clear memory every 5 images
            if i % 5 == 0: gc.collect()

        except Exception as e:
            print(f"\n⚠️ Error on {f_name}: {e}")
            continue

    # --- Phase 2: Scoring the Cache ---
    print("\n\n📊 Calculating Final Metrics...")
    stats = {"company": {"f": [], "e": []}, "date": {"f": [], "e": []}, "address": {"f": [], "e": []}, "total": {"f": [], "e": []}}

    for item in results_cache:
        try:
            base = os.path.splitext(item['file'])[0]
            with open(os.path.join(ENT_DIR, base + ".txt"), 'r') as f:
                gt = json.load(f)
            
            raw = item['pred']
            pred = json.loads(raw[raw.find('{'):raw.rfind('}')+1])

            for field in stats.keys():
                p, g = str(pred.get(field, "")).strip().upper(), str(gt.get(field, "")).strip().upper()
                stats[field]["f"].append(fuzz.token_sort_ratio(p, g) / 100.0)
                stats[field]["e"].append(1 if p == g else 0)
        except: continue

    final = []
    for k, v in stats.items():
        final.append({"Field": k.capitalize(), "Fuzzy": sum(v['f'])/len(v['f']), "EM": sum(v['e'])/len(v['e'])})
    
    print("\n" + "="*50)
    print(pd.DataFrame(final).to_string(index=False, formatters={'Fuzzy': '{:,.2%}'.format, 'EM': '{:,.2%}'.format}))
    print("="*50)

if __name__ == "__main__":
    run_evaluation()

🚀 Starting Resilient Phase 1 (512px Optimized)
[2/100] Processing: X00016469671.jpg...

KeyboardInterrupt: 